In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

# 1. Define your Unity Catalog paths
# Replace 'landing_zone' and the filename with your exact volume details
source_volume_path = "/Volumes/hfal/source/data_source/PHY_R26_P05_V10_D24_Prov_Svc.csv" 
bronze_table_name = "hfal.bronze.medicare_physician_claims_2024"

# 2. Read the raw data from the Source Volume
# PySpark automatically infers the schema (mapping your NPIs, HCPCS codes, and financial metrics)
raw_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(source_volume_path)

# 3. Add Data Engineering Lineage (Crucial for Bronze Tables)
# This tracks exactly when the record was ingested and from which file
bronze_df = raw_df \
    .withColumn("bronze_ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file_path", col("_metadata.file_path"))

# 4. Write to the Bronze Schema as a Managed Delta Table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_table_name)

print(f"Success! Data loaded into Bronze Delta Table: {bronze_table_name}")